In [1]:
# !pip install langchain-chroma langchain-ollama
# 引入套件
import os
from langchain_unstructured import UnstructuredLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_ollama import OllamaEmbeddings
from langchain_chroma import Chroma

# 文件分塊
loader = UnstructuredLoader('data/1.txt')
docs = loader.load()
text_splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=50)
all_splits = text_splitter.split_documents(docs)
# 分成幾塊(Chunks)
print('塊數： ', len(all_splits))
print(all_splits)

塊數：  10
[Document(metadata={'source': 'data/1.txt', 'last_modified': '2026-09-19T11:43:38', 'languages': ['kor', 'zho'], 'file_directory': 'data', 'filename': '1.txt', 'filetype': 'text/plain', 'category': 'Title', 'element_id': '94695db5b74e1bc351ffd7f937fdeddc'}, page_content='黃金價格週五（18日）升至一週高點，並在連續4週下跌後首次錄得單週上漲，因油價下跌緩解了市場對通膨壓力持續時間過長的擔憂。'), Document(metadata={'source': 'data/1.txt', 'last_modified': '2026-09-19T11:43:38', 'languages': ['kor', 'zho'], 'file_directory': 'data', 'filename': '1.txt', 'filetype': 'text/plain', 'category': 'Title', 'element_id': '557aa16bdec1751ce0e60e94c99c75ad'}, page_content='現貨黃金上漲1.2%，報每盎司4390.11美元，盤中一度觸及9月11日以來最高水準。本週迄今上漲1%。'), Document(metadata={'source': 'data/1.txt', 'last_modified': '2026-09-19T11:43:38', 'languages': ['kor', 'zho'], 'file_directory': 'data', 'filename': '1.txt', 'filetype': 'text/plain', 'category': 'Title', 'element_id': '94dc8fde86152465a2614ab62eaddb44'}, page_content='美國黃金期貨結算價上漲0.6%，報每盎司4424.90美元。'), Document(metadata={'sou

In [2]:
# Chroma 只允許字串，但 split_documents 函數會產生 List，故修改languages值為字串
for index in range(len(all_splits)):
        all_splits[index].metadata['languages'] = 'zh'

In [3]:
# 使用 Ollama 載入詞嵌入模型
from langchain.embeddings import init_embeddings
embeddings = init_embeddings(model="qwen3-embedding", provider = 'ollama')

# 使用存於記憶體的向量資料庫
vector_store = Chroma(
    collection_name="my_collection",
    embedding_function=embeddings,
    persist_directory="./chroma_db",
)

# 將所有區塊存入向量資料庫
id_list = [f'黃金_{index+1:0d}' for index, _ in enumerate(all_splits)]  
_ = vector_store.add_documents(ids=id_list, documents=all_splits)

INFO: Anonymized telemetry enabled. See                     https://docs.trychroma.com/telemetry for more information.
INFO: HTTP Request: POST http://127.0.0.1:11434/api/embed "HTTP/1.1 200 OK"


In [4]:
# 向量資料庫內容
result = vector_store.get()
print('向量資料庫內容:')
print(len(result['ids']))
print(result)

{'ids': ['黃金_1', '黃金_2', '黃金_3', '黃金_4', '黃金_5', '黃金_6', '黃金_7', '黃金_8', '黃金_9', '黃金_10'], 'embeddings': None, 'documents': ['黃金價格週五（18日）升至一週高點，並在連續4週下跌後首次錄得單週上漲，因油價下跌緩解了市場對通膨壓力持續時間過長的擔憂。', '現貨黃金上漲1.2%，報每盎司4390.11美元，盤中一度觸及9月11日以來最高水準。本週迄今上漲1%。', '美國黃金期貨結算價上漲0.6%，報每盎司4424.90美元。', 'EverBank全球市場總裁加夫尼（Chris Gaffney）表示：「油價走低降低了通膨壓力，因為石油一直是整體通膨的主要推動因素，貴金屬投資人原本預期美國升息，因此建立大量空頭部位，以利用預期中的黃金拋售。這些部位已迅速解除。」', '美元升至逾7週高點，使以美元計價的黃金對持有其他貨幣的投資人而言價格更高。', '美國聯準會週三將利率上調0.25個百分點，至3.75%至4%的區間，並釋放未來幾個月將進一步升息的訊號。', '根據CME FedWatch工具，交易員目前認為，聯準會官員10月下次開會時再次升息的機率為55%。', '儘管黃金傳統上被視為通膨避險工具，但較高的利率可能降低其吸引力，因為這會讓能產生收益的資產更具吸引力。', '加夫尼認為：「金價目前正在測試4,400至4,440美元附近的阻力區間，而突破這個阻力位可能為金價進一步上漲打開道路。」', '其他貴金屬方面，現貨白銀上漲2.3%，報每盎司66.70美元；白金上漲2.2%，報1,812.50美元；鈀金上漲1.5%，報1,310.20美元。所有金屬均錄得單週上漲。'], 'uris': None, 'included': ['metadatas', 'documents'], 'data': None, 'metadatas': [{'last_modified': '2026-09-19T11:43:38', 'id': 0, 'filename': '1.txt', 'languages': 'zh', 'file_directory': 'data', 'source': 'data/1.txt', 'category': 'Title

In [5]:
# 進行相似度搜尋(Similarity Search)
query = "最新黃金價格=?"
results = vector_store.similarity_search(query, k=2, filter={"source": 'data/1.txt'})
for res in results:
    print(f"* {res.page_content} [{res.metadata}]")

INFO: HTTP Request: POST http://127.0.0.1:11434/api/embed "HTTP/1.1 200 OK"


* 現貨黃金上漲1.2%，報每盎司4390.11美元，盤中一度觸及9月11日以來最高水準。本週迄今上漲1%。 [{'category': 'Title', 'filetype': 'text/plain', 'element_id': '557aa16bdec1751ce0e60e94c99c75ad', 'source': 'data/1.txt', 'last_modified': '2026-09-19T11:43:38', 'id': 1, 'languages': 'zh', 'file_directory': 'data', 'filename': '1.txt'}]
* 美國黃金期貨結算價上漲0.6%，報每盎司4424.90美元。 [{'languages': 'zh', 'filetype': 'text/plain', 'filename': '1.txt', 'element_id': '94dc8fde86152465a2614ab62eaddb44', 'category': 'Title', 'last_modified': '2026-09-19T11:43:38', 'source': 'data/1.txt', 'file_directory': 'data', 'id': 2}]


In [6]:
# 以詞向量進行相似度搜尋
results = vector_store.similarity_search_by_vector(
    embedding=embeddings.embed_query(query), k=1
)
for res in results:
    print(f"* {res.page_content} [{res.metadata}]")

INFO: HTTP Request: POST http://127.0.0.1:11434/api/embed "HTTP/1.1 200 OK"


* 現貨黃金上漲1.2%，報每盎司4390.11美元，盤中一度觸及9月11日以來最高水準。本週迄今上漲1%。 [{'category': 'Title', 'filetype': 'text/plain', 'id': 1, 'source': 'data/1.txt', 'languages': 'zh', 'file_directory': 'data', 'element_id': '557aa16bdec1751ce0e60e94c99c75ad', 'filename': '1.txt', 'last_modified': '2026-09-19T11:43:38'}]


In [7]:
# 更新第一個區塊內容
all_splits[0].page_content = '黃金突然大漲！衝上1週高點 關鍵推手曝光\n' +  all_splits[0].page_content
_ = vector_store.update_documents(ids=id_list[0], documents=[all_splits[0]])

# 刪除最後一個區塊
_ = vector_store.delete(ids=id_list[-1])

INFO: HTTP Request: POST http://127.0.0.1:11434/api/embed "HTTP/1.1 200 OK"


In [8]:
result = vector_store.get()
print('更新後內容:')
print(len(result['ids']))
print(result['documents'])  

['黃金突然大漲！衝上1週高點 關鍵推手曝光\n黃金價格週五（18日）升至一週高點，並在連續4週下跌後首次錄得單週上漲，因油價下跌緩解了市場對通膨壓力持續時間過長的擔憂。', '現貨黃金上漲1.2%，報每盎司4390.11美元，盤中一度觸及9月11日以來最高水準。本週迄今上漲1%。', '美國黃金期貨結算價上漲0.6%，報每盎司4424.90美元。', 'EverBank全球市場總裁加夫尼（Chris Gaffney）表示：「油價走低降低了通膨壓力，因為石油一直是整體通膨的主要推動因素，貴金屬投資人原本預期美國升息，因此建立大量空頭部位，以利用預期中的黃金拋售。這些部位已迅速解除。」', '美元升至逾7週高點，使以美元計價的黃金對持有其他貨幣的投資人而言價格更高。', '美國聯準會週三將利率上調0.25個百分點，至3.75%至4%的區間，並釋放未來幾個月將進一步升息的訊號。', '根據CME FedWatch工具，交易員目前認為，聯準會官員10月下次開會時再次升息的機率為55%。', '儘管黃金傳統上被視為通膨避險工具，但較高的利率可能降低其吸引力，因為這會讓能產生收益的資產更具吸引力。', '加夫尼認為：「金價目前正在測試4,400至4,440美元附近的阻力區間，而突破這個阻力位可能為金價進一步上漲打開道路。」']
